# Día 5 · Unidad 5 — POO aplicada (II): properties, herencia y revisión crítica de código con IA

**Objetivos de aprendizaje**
- Usar `@property` y su `@setter` para exponer atributos calculados o validados sin cambiar la forma en que se usan.
- Aplicar herencia básica entre clases de pólizas, usando `super().__init__()` correctamente.
- Reconocer anti-patrones habituales de POO.
- **Revisar críticamente un fragmento de código con un fallo sutil** — el bloque de IA más importante del día: encontrar un error de lógica que no lanza ninguna excepción, sin resolverlo preguntándole a otra IA.

**Duración estimada:** 75-90 minutos.

**Requisitos previos:** `01_teoria_clases_atributos_metodos.ipynb` de este mismo día.

## 1. `@property` y `@setter`

Ya viste que se puede acceder y modificar un atributo directamente (`poliza.prima = 100`). El problema es que así no hay forma de **validar** el nuevo valor, ni de exponer un dato que en realidad se **calcula** a partir de otros.

`@property` permite que un método se comporte como si fuera un atributo — se lee sin paréntesis — mientras por dentro ejecuta código.

In [1]:
class Poliza:
    def __init__(self, asegurado: str, prima_base: float):
        self.asegurado = asegurado
        self._prima_base = prima_base  # convención: guion bajo = "de uso interno"

    @property
    def prima_base(self) -> float:
        return self._prima_base

    @prima_base.setter
    def prima_base(self, nueva_prima_base: float) -> None:
        if nueva_prima_base < 0:
            raise ValueError("La prima base no puede ser negativa")
        self._prima_base = nueva_prima_base

    @property
    def prima_con_impuestos(self) -> float:
        """Prima con el 8% de Impuesto sobre Primas de Seguro (IPS) aplicado.

        No se guarda en ningún atributo: se recalcula cada vez que se consulta.
        """
        return round(self._prima_base * 1.08, 2)


poliza = Poliza(asegurado="Anna Puig", prima_base=500.0)
print(poliza.prima_con_impuestos)   # 540.0

poliza.prima_base = 600.0           # usa el setter, valida y actualiza _prima_base
print(poliza.prima_con_impuestos)   # 648.0: se recalcula solo, porque es una property

540.0
648.0


Fíjate en el detalle clave: `prima_con_impuestos` **nunca se guarda**. Cada vez que escribes `poliza.prima_con_impuestos`, Python ejecuta el método y calcula el valor en ese momento a partir de `self._prima_base`, que es la fuente de verdad. Por eso, al cambiar `prima_base`, `prima_con_impuestos` "se entera" automáticamente.

> 📊 En Excel esto sería como una columna con una fórmula (`=B2*1.08`) frente a una columna con un valor pegado como texto: la fórmula se recalcula sola cuando cambias `B2`; el valor pegado se queda congelado aunque cambies el resto de la fila.

Intentar poner un valor sin `@setter` falla, porque la property de solo lectura no acepta asignación:

In [2]:
class PolizaSoloLectura:
    def __init__(self, prima_base: float):
        self._prima_base = prima_base

    @property
    def prima_base(self) -> float:
        return self._prima_base


poliza_ro = PolizaSoloLectura(prima_base=500.0)
try:
    poliza_ro.prima_base = 999.0  # no hay @setter -> error
except AttributeError as e:
    print("Error esperado:", e)

Error esperado: property 'prima_base' of 'PolizaSoloLectura' object has no setter


> ⚠️ **Error típico**: escribir el método sin `self`, o llamar a la property con paréntesis (`poliza.prima_con_impuestos()`) como si fuera un método normal — con `@property` se accede **sin** paréntesis, exactamente como a un atributo.

**Para ti:** añade a `Poliza` una property `esta_domiciliada` calculada a partir de un nuevo atributo `_forma_pago` (`"domiciliacion"` o `"transferencia"`), que devuelva `True`/`False`. No necesitas `@setter` para esta.

## 2. Herencia básica

La herencia permite que una clase adquiera atributos y métodos de otra, y añada o modifique lo que necesite. Es útil cuando varios tipos de póliza comparten una base común (asegurado, prima, vigencia) pero cada una tiene sus propios datos.

In [3]:
# Tu código aquí (property "esta_domiciliada")

In [4]:
class Poliza:
    def __init__(self, asegurado: str, prima_base: float):
        self.asegurado = asegurado
        self.prima_base = prima_base
        self.vigente = True

    def anular(self) -> None:
        self.vigente = False

    def __repr__(self) -> str:
        return f"{type(self).__name__}(asegurado={self.asegurado!r}, prima_base={self.prima_base})"


class PolizaAuto(Poliza):
    def __init__(self, asegurado: str, prima_base: float, matricula: str):
        super().__init__(asegurado, prima_base)  # inicializa lo que ya sabe hacer Poliza
        self.matricula = matricula

    def cambiar_matricula(self, nueva_matricula: str) -> None:
        self.matricula = nueva_matricula


class PolizaHogar(Poliza):
    def __init__(self, asegurado: str, prima_base: float, valor_continente: float):
        super().__init__(asegurado, prima_base)
        self.valor_continente = valor_continente


poliza_auto = PolizaAuto(asegurado="Anna Puig", prima_base=480.0, matricula="1234-ABC")
poliza_hogar = PolizaHogar(asegurado="Marc Soler", prima_base=210.0, valor_continente=180_000)

print(poliza_auto)
print(poliza_auto.vigente, poliza_auto.matricula)   # vigente heredado de Poliza, matricula propio

poliza_auto.anular()   # metodo heredado de Poliza, funciona igual en la subclase
print(poliza_auto.vigente)

PolizaAuto(asegurado='Anna Puig', prima_base=480.0)
True 1234-ABC
False


`super().__init__(...)` llama al `__init__` de la clase padre para que haga su parte (`asegurado`, `prima_base`, `vigente`) antes de que la subclase añada lo suyo (`matricula`, `valor_continente`). `PolizaAuto` y `PolizaHogar` heredan `anular()` sin tener que reescribirlo.

> ⚠️ **Error típico**: sobreescribir `__init__` en la subclase **sin** llamar a `super().__init__()`. El objeto se crea sin lanzar ningún error, pero le faltan los atributos que debía inicializar la clase padre — hasta que alguien intenta usarlos.

In [5]:
class PolizaAutoConBug(Poliza):
    def __init__(self, asegurado: str, prima_base: float, matricula: str):
        # Fallo: no se llama a super().__init__(), así que "vigente" nunca se crea
        self.asegurado = asegurado
        self.prima_base = prima_base
        self.matricula = matricula


poliza_con_bug = PolizaAutoConBug(asegurado="Anna Puig", prima_base=480.0, matricula="1234-ABC")
print(poliza_con_bug.asegurado, poliza_con_bug.matricula)   # esto sí existe

try:
    print(poliza_con_bug.vigente)   # nunca se inicializó
except AttributeError as e:
    print("Error esperado:", e)

Anna Puig 1234-ABC
Error esperado: 'PolizaAutoConBug' object has no attribute 'vigente'


La instancia se crea sin problema — el error solo aparece más tarde, cuando algún otro método (por ejemplo, uno que compruebe `if poliza.vigente:`) intenta usar un atributo que nunca llegó a existir. Es exactamente el tipo de fallo que interesa en este notebook: no salta al ejecutar, salta después, en un punto alejado de la causa real.

## 3. Anti-patrones a evitar

La POO es potente, pero mal usada complica el mantenimiento. Tres olores de código (*code smells*) habituales, con más detalle en [refactoring.guru](https://refactoring.guru/refactoring/smells):

- **[Bloaters](https://refactoring.guru/refactoring/smells/bloaters)**: clases que han crecido tanto que son difíciles de entender de un vistazo — una `Poliza` que acaba con 40 atributos y 30 métodos porque todo se le fue añadiendo ahí.
- **[Object-Orientation Abusers](https://refactoring.guru/refactoring/smells/oo-abusers)**: uso incorrecto de herencia — por ejemplo, heredar solo para reutilizar un método, aunque conceptualmente las clases no tengan relación "es un tipo de".
- **[Couplers](https://refactoring.guru/refactoring/smells/couplers)**: clases excesivamente acopladas entre sí, de forma que cambiar una obliga a tocar otras varias.

Si te encuentras dudando entre herencia y una relación más simple (una clase que *usa* a otra como atributo, en vez de heredar de ella), suele ser buena señal mirar el principio [*composition over inheritance*](https://en.wikipedia.org/wiki/Composition_over_inheritance). No hace falta profundizar más en este curso, pero conviene saber que existe el debate.

## 4. Revisión crítica de código generado por IA — el bloque más importante del día

Recupera un momento `docs/uso_ia_en_el_curso.md`: *"el código generado puede ejecutarse sin error y aun así no calcular lo que le habéis pedido"*. Este es exactamente ese caso, y por eso es el bloque de IA de la unidad de POO en vez del formato habitual de "prompt sugerido".

A continuación tienes un fragmento de código construido para el curso — imagina que es la respuesta de un asistente de IA a un prompt como *"crea una clase de póliza con un descuento del 5% si el pago está domiciliado"*. **Antes de seguir bajando**, léelo con calma e intenta responder: ¿hace lo que promete? ¿Qué pasaría si, después de crear la póliza, alguien corrige la prima base?

In [6]:
class PolizaConDescuento:
    """Póliza con un descuento del 5% si el pago está domiciliado."""

    def __init__(self, asegurado: str, prima_base: float, domiciliada: bool):
        self.asegurado = asegurado
        self._prima_base = prima_base
        self.domiciliada = domiciliada
        descuento = 0.05 if domiciliada else 0.0
        self.prima_final = round(self._prima_base * (1 - descuento), 2)

    @property
    def prima_base(self) -> float:
        return self._prima_base

    @prima_base.setter
    def prima_base(self, nueva_prima_base: float) -> None:
        """Permite corregir la prima base, p. ej. tras una regularización."""
        self._prima_base = nueva_prima_base

**Para ti, antes de ejecutar la siguiente celda:** ¿qué esperas que imprima `poliza.prima_final` después de la línea `poliza.prima_base = 800.0`? Escribe tu predicción (aunque sea mentalmente) antes de seguir.

In [7]:
poliza = PolizaConDescuento(asegurado="Anna Puig", prima_base=500.0, domiciliada=True)
print("Prima final al crear la póliza:", poliza.prima_final)   # 475.0 = 500 * 0.95, correcto

poliza.prima_base = 800.0   # se corrige la prima base tras una regularización
print("Nueva prima_base:", poliza.prima_base)                  # 800.0, el setter sí la actualiza
print("Prima final tras corregir la prima base:", poliza.prima_final)   # ¿sigue siendo 475.0?

Prima final al crear la póliza: 475.0
Nueva prima_base: 800.0
Prima final tras corregir la prima base: 475.0


### El fallo

`poliza.prima_final` **no se actualiza**. Sigue mostrando `475.0` aunque la prima base ya es `800.0` (el resultado correcto sería `760.0` = `800 * 0.95`). No hay ningún error, ninguna excepción, ningún *warning* — el código "funciona" perfectamente y da un número creíble. Ese es justo el tipo de fallo más peligroso: uno que no se nota si no se compara contra un cálculo hecho a mano.

**La causa**: `prima_final` se calcula **una sola vez**, dentro de `__init__`, y se guarda como un atributo normal. El `@setter` de `prima_base` actualiza `self._prima_base` correctamente, pero nadie le dijo que también tenía que recalcular `prima_final` — son dos atributos independientes que deberían estar sincronizados y no lo están.

### La corrección

La solución es la misma idea que ya viste con `prima_con_impuestos`: convertir `prima_final` en una **property calculada**, para que se recalcule cada vez que se consulta, en vez de guardarse como un valor congelado en el tiempo.

In [8]:
class PolizaConDescuentoCorregida:
    """Póliza con un descuento del 5% si el pago está domiciliado."""

    def __init__(self, asegurado: str, prima_base: float, domiciliada: bool):
        self.asegurado = asegurado
        self._prima_base = prima_base
        self.domiciliada = domiciliada

    @property
    def prima_base(self) -> float:
        return self._prima_base

    @prima_base.setter
    def prima_base(self, nueva_prima_base: float) -> None:
        self._prima_base = nueva_prima_base

    @property
    def prima_final(self) -> float:
        """Se recalcula cada vez que se consulta: nunca puede quedar desincronizada."""
        descuento = 0.05 if self.domiciliada else 0.0
        return round(self._prima_base * (1 - descuento), 2)


poliza_ok = PolizaConDescuentoCorregida(asegurado="Anna Puig", prima_base=500.0, domiciliada=True)
print("Prima final al crear la póliza:", poliza_ok.prima_final)   # 475.0

poliza_ok.prima_base = 800.0
print("Prima final tras corregir la prima base:", poliza_ok.prima_final)   # 760.0, correcto

Prima final al crear la póliza: 475.0
Prima final tras corregir la prima base: 760.0


### Por qué importa para vuestro trabajo real

Nada en el fragmento original "olía" a error: se ejecuta, no lanza ninguna excepción, y el primer resultado (`475.0`) es correcto — lo que lo hace fácil de dar por bueno en una revisión superficial. Solo aparece al combinar dos operaciones en un orden concreto (crear la póliza y **después** corregir la prima), algo que una prueba rápida con un único caso no habría detectado.

Es el mismo tipo de fallo que un LLM puede introducir en un modelo actuarial: un atributo derivado (una provisión, una prima recalculada, un ratio) que se calcula una vez y no se sincroniza cuando cambia algo de lo que depende. La única defensa es la misma de siempre: **no dar el código por bueno porque se ejecute sin error**, sino comprobar el resultado contra un caso calculado a mano — y, para este ejercicio en concreto, hacerlo sin pedirle la respuesta a otro asistente de IA: el objetivo es entrenar tu propia lectura crítica, no la de otra IA (ver `docs/uso_ia_en_el_curso.md`).

## Resumen

- `@property` expone un método como si fuera un atributo (se lee sin paréntesis); `@nombre.setter` permite validar o transformar lo que se le asigna.
- Una property **calculada** (sin atributo propio detrás) se recalcula cada vez que se consulta — como una fórmula de Excel — y por eso nunca puede quedar desincronizada de los datos de los que depende.
- La herencia (`class Hija(Padre):`) reutiliza atributos y métodos; `super().__init__(...)` debe llamarse siempre que la subclase defina su propio `__init__`, o los atributos de la clase padre nunca se crean.
- Olvidar `super().__init__()` no da error al crear el objeto — el error aparece después, al intentar usar un atributo que nunca se inicializó.
- Tres anti-patrones a vigilar: clases sobrecargadas (*bloaters*), abuso de herencia, y acoplamiento excesivo entre clases.
- **El fallo más peligroso no es el que lanza una excepción, es el que calcula algo incorrecto en silencio.** Un atributo derivado guardado una sola vez, en vez de recalculado, es un ejemplo clásico — y es exactamente el tipo de error que hay que aprender a detectar leyendo con criterio, no confiando en que "si se ejecuta, está bien".
- Nunca resuelvas un ejercicio de "encuentra el fallo" preguntándole la respuesta a un asistente de IA: pierde su sentido pedagógico.

**Siguiente:** `03_teoria_dataclasses_enum.ipynb`, donde veremos `dataclasses` y `Enum`, y cerraremos con una mención opcional a magic methods avanzados y patrones de diseño.